In [ ]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score


In [ ]:
%pip install xgboost imbalanced-learn

In [ ]:
df= pd.read_csv('credit fraud.csv')
df

In [ ]:
#Data Preprocessing (MUD)

#1-Missing values= 0 
print(df.isna().sum())

#2-Unique values
print(df.nunique())
#df = df.drop (columns=["zipcodeOri", "zipMerchant"])
print(df.columns)# to check

#3-Duplicates = 0
print("Duplicate rows:", df.duplicated().sum())
print(df.shape)

In [ ]:
#Data Imbalance detection
print(df["fraud"].value_counts())
print(df["fraud"].value_counts(normalize=True) * 100)

sns.countplot(x="fraud", data=df)
plt.title("Fraud (1) vs Normal (0)")
plt.show()

In [ ]:
# Amount vs fraud
df.groupby("fraud")["amount"].mean().plot(kind="bar", color=["steelblue", "tomato"])
plt.xticks([0, 1], ["Normal", "Fraud"], rotation=0)
plt.title("Average Amount of Fraud")
plt.show()

# Fraud rate in each category category
df.groupby("category")["fraud"].mean().plot(kind="bar")
plt.title("Fraud Rate in each Category")
plt.show()

# Fraud rate by age
df.groupby("age")["fraud"].mean().plot(kind="bar")
plt.title("Fraud Rate by Age")
plt.show()

# Fraud rate by gender
df.groupby("gender")["fraud"].mean().plot(kind="bar")
plt.title("Fraud Rate by Gender")
plt.show()

# number of frauds per category
category = df.groupby("category")["fraud"].agg(["mean", "count", "sum"])
category.columns = ["Fraud Rate", "Transactions", "Frauds"]
category["Fraud Rate"] = (category["Fraud Rate"] * 100).round(1)
category = category.sort_values("Fraud Rate", ascending=False)
print(category)

print(df.groupby("fraud")["amount"].mean())
print(df.groupby("category"))
print(df.groupby("age")["fraud"].mean())
print(df.groupby("gender")["fraud"].mean())

In [ ]:
df_encoded = pd.get_dummies(df, columns=["age", "gender", "category", "merchant"], dtype=int)
print(df_encoded.shape)
print(df_encoded.columns.tolist())

In [ ]:
df = df.sort_values("step").reset_index(drop=True)
df["hour_of_day"] = df["step"] % 24
print(df.groupby("fraud")["hour_of_day"].mean())


In [ ]:
x = df_encoded.drop(columns=["fraud", "customer", "step"])
y = df_encoded["fraud"]
#print(x.shape, y.shape)
#print(y.value_counts())
#print(x.value_counts())
print(list(x.columns))
print(len(x.columns))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
from sklearn.preprocessing import RobustScaler

num_cols = ["amount", "hour_of_day"]
scaler = RobustScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
#With Imabalanced data
lr = LogisticRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
print("Logistic Regression  F1:", f1_score(y_test, lr_pred), " ROC-AUC:", roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1]))

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("Random Forest F1:", f1_score(y_test, rf_pred)," ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

In [ ]:
xgb = XGBClassifier(n_estimators=300, random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
print("XGBoost F1:", f1_score(y_test, xgb_pred)," ROC-AUC:", roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]))

In [ ]:
smote = SMOTE(sampling_strategy='minority', random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
pd.Series(y_train_sm).value_counts().plot(kind="bar", color=["steelblue", "tomato"])
plt.xticks([0,1], ['Normal', 'Fraud'])
plt.ylabel("Count")
plt.title("Class Distribution after smote")
plt.show()

In [ ]:
lr_smote = LogisticRegression()
lr_smote.fit(X_train_sm, y_train_sm)
lr_smote_pred = lr_smote.predict(X_test)
print("Logistic Regression + SMOTE F1:", f1_score(y_test, lr_smote_pred)," ROC-AUC:", roc_auc_score(y_test, lr_smote.predict_proba(X_test)[:, 1]))

In [ ]:
rf_smote= RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_sm, y_train_sm)
rf_smote_pred= rf_smote.predict(X_test)
print("Random Forest+SMOTE F1:",f1_score(y_test, rf_smote_pred),"ROC-AUC:", roc_auc_score(y_test,rf_smote.predict_proba(X_test)[:, 1]))


In [ ]:
xgb_smote= XGBClassifier(n_estimators=300, random_state=42)
xgb_smote.fit(X_train_sm, y_train_sm)
xgb_smote_pred= xgb_smote.predict(X_test)
print("XGBoost+SMOTE F1:",f1_score(y_test, xgb_smote_pred),"ROC-AUC:",roc_auc_score(y_test,xgb_smote.predict_proba(X_test)[:, 1]))

In [ ]:
results = pd.DataFrame({"Original":[f1_score(y_test, lr_pred),f1_score(y_test, rf_pred),f1_score(y_test, xgb_pred)],"SMOTE":[f1_score(y_test, lr_smote_pred),f1_score(y_test, rf_smote_pred), f1_score(y_test, xgb_smote_pred)],},index=["Logistic Regression", "Random Forest", "XGBoost"])
results.round(3)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
params={"n_estimators": [200, 300, 500],"max_depth": [4, 6, 8],"learning_rate": [0.05, 0.1, 0.2],}
search=RandomizedSearchCV(XGBClassifier(),params,n_iter=10,scoring="f1",cv=3)
search.fit(X_train, y_train)
best_model=search.best_estimator_
print(search.best_params_)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
best_prid = best_model.predict(X_test)
print("F1:", f1_score(y_test,best_prid))
print("ROC-AUC:", roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1]))
print(classification_report(y_test,best_prid))
ConfusionMatrixDisplay(confusion_matrix(y_test,best_prid),display_labels=["Normal","Fraud"]).plot(cmap="Blues")
plt.title("Confusion Matrix-Tuned XGBoost")
plt.show()

In [ ]:
best_model.set_params(importance_type="gain")
importance = pd.Series(best_model.feature_importances_,index=X.columns).sort_values(ascending=False)
importance.head(20).sort_values().plot(kind="barh")
plt.title("Top 10 Features (Gain)")
plt.show()